# Benchmarking ECDLP functions

In [1]:
import qrisp
import numpy as np
import src.classical.ec_arithmetic as clECarithm
import src.quantum.ec_arithmetic as qECarithm

In [2]:
def benchmark_circuit(res):
    
    qubit_count = len([qb for qb in res.qs.qubits if qb.allocated])

    compiled_circuit = res.qs.compile()

    gate_counts = compiled_circuit.transpile().count_ops()
    print(gate_counts)
    cnot_count = gate_counts.get('cx', 0)
    t_count = gate_counts.get('t', 0)
    t_depth = compiled_circuit.depth()  # Assumes depth of T gates aligns with circuit depth

    return {
        "qubit_count": qubit_count,
        "t_count": t_count,
        "cnot_count": cnot_count,
        "t_depth": t_depth
    }

In [13]:
p=7
a=5
b=4
curve = clECarithm.EllCurve(a, b, p)

In [ ]:
v = qrisp.QuantumModulus(p, inpl_adder=gidney_adder)
v[:] = 4
m = qrisp.QuantumArray(qtype=qrisp.QuantumBool(), shape=(2 * p.bit_length(),))

res_kaliski = qECarithm.kaliski_quantum(v, p, m)
for a in m:
    a.delete()

In [15]:
bm = benchmark_circuit(res_kaliski)
print(bm)

{'x': 253, 'cx': 7843, 'h': 1652, 'p': 7994}
{'qubit_count': 3, 't_count': 0, 'cnot_count': 7843, 't_depth': 6315}


In [16]:
x1 = qrisp.QuantumModulus(2**(p.bit_length())) 
x1[:] = 1
mod_p = qrisp.QuantumModulus(p)
anc = qrisp.QuantumArray(qtype=mod_p, shape=(2,))
anc[:] = [2,6]

G = [0,5]

res_ec_add = qECarithm.qrisp_ell_add_inpl(anc, G, p)

In [17]:
bm = benchmark_circuit(res_ec_add[0])
print(bm)

{'x': 1323, 'cx': 50216, 'h': 10190, 'p': 49395, 'cz': 327}
{'qubit_count': 6, 't_count': 0, 'cnot_count': 50216, 't_depth': 38243}


In [18]:
x1 = qrisp.QuantumModulus(2**(p.bit_length())) 
x1[:] = 1
mod_p = qrisp.QuantumModulus(p)
anc = qrisp.QuantumArray(qtype=mod_p, shape=(2,))
anc[:] = [2,6]

G = [0,5]

res_ec_ctrl_add = qECarithm.qrisp_ell_mult_add(G,anc,x1,curve)

In [ ]:
bm = benchmark_circuit(res_ec_ctrl_add[0])
print(bm)

In [20]:
benchmark_results = {}

# Benchmark each resulting circuit individually
benchmark_results["kaliski_quantum"] = benchmark_circuit(res_kaliski)
benchmark_results["ec_addition"] = benchmark_circuit(res_ec_add[0])
#benchmark_results["controlled_addition"] = benchmark_circuit(res_ec_ctrl_add[0])

{'x': 253, 'cx': 7843, 'h': 1652, 'p': 7994}
{'x': 1323, 'cx': 50216, 'h': 10190, 'p': 49395, 'cz': 327}


In [22]:
import pandas as pd

# Display results in a tabular format
results_df = pd.DataFrame(benchmark_results).T
print(results_df)

ModuleNotFoundError: No module named 'pandas'